# Text Classification Pipeline

This notebook implements the training and testing classification pipeline. It generates category descriptions using Gemini models and classifies a test split to produce raw outputs for further analysis.

## Workflow:
1. Load data from JSON (categories and texts).
2. Split data 80% training / 20% testing.
3. Generate descriptions using Gemini 3.1 Flash lite.
4. Classify test data in randomized batches of 20.
5. Ablation study with Gemini 3.5 Flash.

## 1. Setup and Configuration
Install necessary libraries and configure the Gemini API.

In [4]:
!pip install -q google-genai pandas numpy matplotlib seaborn scikit-learn scipy sentence-transformers

import os
import json
import random
import pandas as pd
import numpy as np
from google import genai
from sklearn.model_selection import train_test_split
from datetime import datetime

from google.colab import userdata

# Retrieve the secret from Colab's secret manager
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = api_key
except userdata.NotebookAccessError:
    print("Please click the key icon on the left and grant notebook access to GOOGLE_API_KEY.")
except userdata.SecretNotFoundError:
    print("The secret GOOGLE_API_KEY was not found. Please add it to the Secrets panel.")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab or drive mount failed.')

# Configuration
CONFIG = {
    'MODEL_31': 'gemini-3.1-flash-lite',
    'MODEL_35': 'gemini-3.5-flash-latest',
    'OUTPUT_DIR': '/content/drive/MyDrive/classification_outputs',
    'REUSE_CACHE': True,
    'API_KEY': os.getenv('GOOGLE_API_KEY')
}

if not os.path.exists(CONFIG['OUTPUT_DIR']):
    try:
        os.makedirs(CONFIG['OUTPUT_DIR'])
    except OSError:
        os.makedirs('outputs', exist_ok=True)
        CONFIG['OUTPUT_DIR'] = 'outputs'

client = genai.Client(api_key=CONFIG['API_KEY'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. LLM Caching Utility
Save and load LLM responses to avoid hitting quotas.

In [5]:
def get_cache_path(name, timestamp=None):
    if timestamp:
        return os.path.join(CONFIG['OUTPUT_DIR'], f'{name}_{timestamp}.json')
    return os.path.join(CONFIG['OUTPUT_DIR'], f'{name}_latest.json')

def save_cache(name, data):
    # Save latest
    latest_path = get_cache_path(name)
    with open(latest_path, 'w') as f:
        json.dump(data, f)

    # Save timestamped copy
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    versioned_path = get_cache_path(name, timestamp=timestamp)
    with open(versioned_path, 'w') as f:
        json.dump(data, f)

def load_cache(name):
    path = get_cache_path(name)
    if CONFIG['REUSE_CACHE'] and os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None


## 3. Data Loading
Load the dataset from Google Drive.

In [6]:
def load_data(file_path):
    if not os.path.exists(file_path):
        print(f'Warning: File not found at {file_path}. Using dummy data.')
        return {
            'technology': ['Computers are fast', 'Software is evolving', 'AI is the future', 'Hardware improves over time'],
            'sports': ['Football is popular', 'Basketball requires skill', 'Tennis is played on grass or clay', 'Running is good exercise'],
            'cooking': ['Baking requires precision', 'Spices add flavor', 'Italian cuisine is famous', 'Vegetables are healthy']
        }
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

DATA_PATH = '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channel_titles_cleaned.json'
data = load_data(DATA_PATH)

## 4. Data Splitting
Split data 80% training and 20% testing per category.

In [7]:
train_data = {}
test_data = {}

for category, texts in data.items():
    train, test = train_test_split(texts, test_size=0.2, random_state=42)
    train_data[category] = train
    test_data[category] = test

print(f'Categories: {list(data.keys())}')

# Save splits for analysis
save_cache('train_data', train_data)
save_cache('test_data', test_data)

Categories: ['20VC with Harry Stebbings', 'ARK Invest', 'Alex Hormozi', 'All-In Podcast', 'Anthony Pompliano', 'Asianometry', 'Bg2 Pod', 'Bloomberg Originals', 'Dan Martell', 'Garry Tan', 'Joe Lonsdale', "Lenny's Podcast", 'My First Million', 'Network State Podcast', 'Patrick Boyle', 'Principles by Ray Dalio', 'Real Vision Presents', 'Sequoia Capital', 'This Week in Startups', 'Tim Ferriss', 'Tony Robbins', 'Y Combinator', 'a16z', 'The Prof G Pod – Scott Galloway']


## 5. Classification Helper Function
A helper function to classify texts in batches using category descriptions.

In [9]:
import time

def classify_texts(texts_with_labels, descriptions, model_name, cache_name):
    results = load_cache(cache_name) or []

    processed_texts = {r['text'] for r in results}
    to_process = [t for t in texts_with_labels if t['text'] not in processed_texts]

    if not to_process:
        print(f'All texts already classified for {cache_name}.')
        return pd.DataFrame(results)

    batch_size = 20
    for i in range(0, len(to_process), batch_size):
        batch = to_process[i:i+batch_size]
        desc_str = '\n'.join([f'{cat}: {desc}' for cat, desc in descriptions.items()])
        texts_to_classify = '\n'.join([f"[{j}] {item['text']}" for j, item in enumerate(batch)])

        prompt = f'Classify the following texts into one of these categories:\n{desc_str}\n\nTexts:\n{texts_to_classify}\n\nReturn only a JSON list of category names in order.'
        time.sleep(5)

        try:
            response = client.models.generate_content(model=model_name, contents=prompt)
            res_text = response.text.strip()
            if '```json' in res_text: res_text = res_text.split('```json')[1].split('```')[0]
            predictions = json.loads(res_text)

            for j, pred in enumerate(predictions):
                if j < len(batch):
                    batch[j]['prediction'] = pred
                    results.append(batch[j])

            save_cache(cache_name, results)
            print(f'Processed batch {i//batch_size + 1}/{(len(to_process)-1)//batch_size + 1}')
        except Exception as e:
            print(f'Error processing batch starting at {i}: {e}')

    return pd.DataFrame(results)

## 6. Iterative Description Generation
Generate descriptions for each category using Gemini 3.1 Flash lite.

In [10]:
import time
descriptions_31 = load_cache('descriptions_31') or {}
valid_categories = list(train_data.keys())

for category in valid_categories:
    if category in descriptions_31:
        continue

    texts = train_data[category]
    prompt = f'Provide a concise description of the following category based on these examples: {category}\nExamples:\n' + '\n'.join(texts)
    response = client.models.generate_content(model=CONFIG['MODEL_31'], contents=prompt)
    descriptions_31[category] = response.text
    save_cache('descriptions_31', descriptions_31)
    print(f'Description for {category} generated.')
    time.sleep(5)

print('Descriptions generated for all valid categories.')

Descriptions generated for all valid categories.


## 7. Batch Classification
Classify test data in randomized batches of 20.

In [11]:
all_test_texts = []
for category in valid_categories:
    texts = test_data[category]
    for t in texts:
        all_test_texts.append({'text': t, 'label': category})

random.seed(42)
random.shuffle(all_test_texts)

valid_descriptions = {cat: descriptions_31[cat] for cat in valid_categories if cat in descriptions_31}
df_results = classify_texts(all_test_texts, valid_descriptions, CONFIG['MODEL_31'], 'classification_results_31')
print('Classification complete on test split.')

Processed batch 1/12
Processed batch 2/12
Processed batch 3/12
Processed batch 4/12
Processed batch 5/12
Processed batch 6/12
Processed batch 7/12
Processed batch 8/12
Processed batch 9/12
Processed batch 10/12
Processed batch 11/12
Processed batch 12/12
Classification complete on test split.


## 8. Ablation Study: Gemini 3.5 Flash (Single Request)
Generate descriptions in a single request and classify.

In [ ]:
ablation_descriptions = load_cache('ablation_descriptions')

if not ablation_descriptions:
    valid_train_data = {cat: texts for cat, texts in train_data.items() if cat in valid_categories}
    all_train_formatted = json.dumps(valid_train_data)
    prompt = f'Generate concise descriptions for ALL the following categories based on their examples:\n{all_train_formatted}\nReturn a JSON object where keys are category names and values are descriptions.'

    response = client.models.generate_content(model=CONFIG['MODEL_35'], contents=prompt)
    res_text = response.text.strip()
    if '```json' in res_text: res_text = res_text.split('```json')[1].split('```')[0]
    ablation_descriptions = json.loads(res_text)
    save_cache('ablation_descriptions', ablation_descriptions)

print('Ablation descriptions generated.')
df_results_ablation = classify_texts(all_test_texts, ablation_descriptions, CONFIG['MODEL_31'], 'classification_results_35')
print('Ablation classification complete.')